# 02 收集编辑：在 CounterFact 的两种关系上做 ROME 编辑并保存结果

**这个 notebook 做什么：**
1. 从 P103（母语）和 P30（大洲）里，筛出原模型本来就知道答案的事实；
2. 逐条做 ROME 编辑，记录**编辑前后**的回答和概率，以及每一层句末 token 的激活差值；
3. 把结果存成文件，后面的分析和 steering 实验都从这些文件开始，不用重复做编辑；
4. 最后一部分做第一个分析：**同一关系内**的编辑差值，是否比**不同关系之间**更相似（用 CounterFact 重做实验 2b）。

**设置**：GPU T4 x2、Internet 打开。

**时间**：每条编辑大约十几到几十秒，两种关系各 200 条，预计 2–4 小时。代码每 20 条自动存一次，会话断了重新运行会从断点继续，不会从头再来。

**和之前实验的区别：**
- 数据换成 CounterFact，数量从 12 条增加到几百条；
- 差值统一用**句末 token**（PhD 学生的建议）；
- 公共函数都放在仓库里的 `steer_utils.py`，notebook 只负责调用。

## 1. 检查 GPU

In [ ]:
!nvidia-smi

## 2. 安装 EasyEdit、下载我们的仓库

**把 `REPO_URL` 改成你自己的仓库地址。** 装完以后**一定要 Run → Restart & clear cell outputs 重启**，再从第 3 步开始。

In [ ]:
REPO_URL = 'https://github.com/<your-username>/thesis-steering.git'   # <-- change this

%cd /kaggle/working
![ -d EasyEdit ] || git clone --depth 1 https://github.com/zjunlp/EasyEdit.git
!rm -rf thesis-steering && git clone --depth 1 {REPO_URL} thesis-steering
!grep -v -E "^(torch|torchvision|PyYAML|sentencepiece|av|opencv-python|numpy|scipy|scikit-learn|pandas|matplotlib|pyarrow)==" EasyEdit/requirements.txt > req.txt
!pip install -q -r req.txt && echo '=== install finished (dependency-conflict messages above can be ignored) ==='

## 3. 加载模型和数据

In [ ]:
import sys, os, time, random
sys.path.insert(0, '/kaggle/working/thesis-steering')
import torch, pandas as pd, matplotlib.pyplot as plt
import steer_utils as su

editor, model, tok, hparams = su.load_editor('/kaggle/working/EasyEdit')
data = su.load_counterfact('/kaggle/working/counterfact.json')

OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
print('model loaded; CounterFact records:', len(data))

## 4. 实验设置

- `RELATIONS`：要用的关系；
- `N_EDITS`：每种关系最多保留多少条**编辑成功**的事实；
- `SEED`：随机种子，保证每次打乱的顺序一样，结果可以复现。

In [ ]:
RELATIONS = ['P103', 'P30']
N_EDITS = 200
SEED = 0

## 5. 筛选：只保留原模型本来就知道的事实

判断标准和实验 3 一样：原模型给原答案的平均对数概率**高于**新答案。模型本来就不知道的事实，没法判断"撤销后有没有回到原答案"，所以要排除。

这一步只做前向计算，几分钟就够。

In [ ]:
known = {}
for rel in RELATIONS:
    recs = su.relation_records(data, rel)
    random.Random(SEED).shuffle(recs)
    t0 = time.time()
    known[rel] = [r for r in recs if su.score_record(model, tok, r)['prefers_true']]
    print(f'{rel}: {len(known[rel])} of {len(recs)} known '
          f'({len(known[rel]) / len(recs):.0%}), {time.time() - t0:.0f}s')

**先想一想**：P30 里大部分原答案是 Antarctica（南极洲）。你觉得 GPT-2 XL 知道的比例会比 P103 高还是低？

## 6. 逐条编辑并保存

每条事实依次：
1. 记录**编辑前**：模型续写的一段文字（8 个 token）、原答案和新答案的对数概率、模型最想说的下一个词（top-1）、每层句末的激活；
2. 做 ROME 编辑；
3. 把权重的改动分解成两根向量 `u`、`v`（ROME 的改动是秩一的，这两根就能完全还原它，比存整个矩阵小得多）；
4. 记录**编辑后**的同样几项，判断编辑是否成功（新答案的概率高于原答案）；
5. 还原权重，并检查还原后回答是否和编辑前一致；
6. 编辑成功的，保存差值（编辑前 − 编辑后，和我们的 steering 向量定义同号）和 `u`、`v`。

结果每 20 条存一次。如果会话中断，重新运行这一格（前面几格也要先跑），会自动跳过已经做过的。

In [ ]:
W = su.rewrite_weight(model, hparams)
W_orig = W.detach().clone()

def store_path(rel):
    return f'{OUT}/edits_{rel}.pt'

for rel in RELATIONS:
    store = torch.load(store_path(rel)) if os.path.exists(store_path(rel)) else dict(rows=[], diff={}, u={}, v={})
    done = {row['case_id'] for row in store['rows']}
    n_ok = sum(row['success'] for row in store['rows'])
    print(f'===== {rel}: {len(done)} already done, {n_ok} successful')
    t0 = time.time()
    for rec in known[rel]:
        if n_ok >= N_EDITS:
            break
        if rec['case_id'] in done:
            continue
        pre = su.score_record(model, tok, rec)
        text_pre = su.generate(model, tok, rec['prompt'], n_tokens=8)
        h_pre = su.hidden_states(model, tok, rec['prompt'])[:, -1]

        su.rome_edit(editor, rec)
        u, v, err = su.rank_one_factors(W.detach() - W_orig)
        post = su.score_record(model, tok, rec)
        text_post = su.generate(model, tok, rec['prompt'], n_tokens=8)
        h_post = su.hidden_states(model, tok, rec['prompt'])[:, -1]

        su.set_edit(W, W_orig)
        restored = su.score_record(model, tok, rec)['top1'] == pre['top1']

        success = post['logp_new'] > post['logp_true']
        store['rows'].append(dict(
            case_id=rec['case_id'], relation=rel, subject=rec['subject'], prompt=rec['prompt'],
            true=rec['true'], new=rec['new'],
            text_before=text_pre, text_after_edit=text_post,
            top1_before=pre['top1'], top1_after_edit=post['top1'],
            logp_true_before=pre['logp_true'], logp_new_before=pre['logp_new'],
            logp_true_after_edit=post['logp_true'], logp_new_after_edit=post['logp_new'],
            success=success, restored=restored, rank1_error=err))
        if success:
            store['diff'][rec['case_id']] = (h_pre - h_post).half()   # [49, 1600]
            store['u'][rec['case_id']], store['v'][rec['case_id']] = u, v
            n_ok += 1
        if len(store['rows']) % 20 == 0:
            torch.save(store, store_path(rel))
            rate = (time.time() - t0) / max(1, len(store['rows']) - len(done))
            print(f'{rel}: {len(store["rows"])} edited, {n_ok} successful, {rate:.1f}s per edit')
    torch.save(store, store_path(rel))
    pd.DataFrame(store['rows']).to_csv(f'{OUT}/edits_{rel}.csv', index=False)
    print(f'{rel} finished: {n_ok} successful edits saved')

## 7. 每条编辑的前后变化

这里有两张表，回答"编辑做了什么"：
- **第一张是具体的文字输出**：同一句话，编辑前和编辑后模型分别续写了什么。这是给人直接看的。
- **第二张是概率**：原答案、新答案在编辑前后的对数概率。统计成功率用的是这张，因为续写的措辞会变（比如说 soccer 而不是 football），只看文字容易误判。

**检查三件事：**
- `success`：大部分应该是 True；
- `restored`：应该全是 True，否则说明权重没还原干净，后面的结果都不可信；
- `rank1_error`：应该非常接近 0，说明用 `u`、`v` 两根向量就能完全还原这条编辑。

In [ ]:
rows = pd.concat([pd.read_csv(f'{OUT}/edits_{rel}.csv') for rel in RELATIONS])
print(rows.groupby('relation')[['success', 'restored']].mean().round(3))
print('max rank-one reconstruction error:', rows.rank1_error.max())
pd.set_option('display.max_colwidth', 50)
print('Generated text before and after the edit:')
display(rows[['relation', 'prompt', 'true', 'new', 'text_before', 'text_after_edit']].head(15))
print('Log-probabilities of the original and the new answer:')
cols = ['relation', 'subject', 'true', 'new', 'logp_true_before', 'logp_new_before',
        'logp_true_after_edit', 'logp_new_after_edit', 'success']
display(rows[cols].head(15).round(2))

## 8. 下载结果

在右侧 **Output** 面板里，把 `results/` 下的文件下载到自己电脑（`.pt` 是差值和编辑向量，`.csv` 是上面那张表）。

后面的 steering 实验也在 Kaggle 上跑，需要用到这些文件。最方便的办法是：在 Kaggle 上新建一个 **Dataset**，把这几个文件传上去，以后的 notebook 直接把它添加为输入。

---

# 分析：同一关系内的编辑，差值是否更相似？

下面几格**不需要 GPU**，可以在这里接着跑，也可以把文件下载后在自己电脑上跑。在本地运行时，把 `OUT` 改成文件所在的文件夹，并确保 `steer_utils.py` 能被导入。

## 运行之前先写下预测

1. 同一关系内（P103 内部、P30 内部）的平均余弦相似度，会比 2b 里的 0.05 高吗？高多少？
2. P30 和 P103 相比，哪个关系内部更相似？为什么？
3. 不同关系之间（P103 vs P30）会是多少？

## 9. 每一层的平均余弦相似度：关系内 vs 关系之间

In [ ]:
D = {}
for rel in RELATIONS:
    store = torch.load(f'{OUT}/edits_{rel}.pt')
    ids = sorted(store['diff'])
    D[rel] = torch.stack([store['diff'][i] for i in ids]).float()   # [n_edits, 49, 1600]
    print(rel, tuple(D[rel].shape))

n_hidden = D[RELATIONS[0]].shape[1]          # 49 for GPT-2 XL
layers = list(range(18, n_hidden))
curves = {f'within {r}': [su.mean_offdiag(su.cos_matrix(D[r][:, L])) for L in layers] for r in RELATIONS}
a, b = RELATIONS
curves[f'{a} vs {b}'] = [su.cos_matrix(D[a][:, L], D[b][:, L]).mean().item() for L in layers]

fig, ax = plt.subplots(figsize=(9, 4))
for name, ys in curves.items():
    ax.plot(layers, ys, marker='o', markersize=3, label=name)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xlabel('hidden_states index (18 = output of edited layer 17)')
ax.set_ylabel('mean pairwise cosine similarity')
ax.set_title('Edit differences (last token): within vs across relations')
ax.legend(); plt.show()
fig.savefig(f'{OUT}/02_within_vs_across.png', dpi=150, bbox_inches='tight')

for name, ys in curves.items():
    best = max(range(len(ys)), key=lambda i: ys[i])
    print(f'{name:14} at index 18: {ys[0]:.3f}   max: {ys[best]:.3f} at index {layers[best]}')

### 讲解
- `D[rel]`：这种关系所有成功编辑的差值，叠成 `[编辑条数, 49, 1600]`。
- `su.cos_matrix(D[r][:, L])`：取第 L 个 hidden state 上所有编辑的差值，算两两之间的余弦相似度，得到一个方阵；`su.mean_offdiag` 去掉对角线后求平均，和 2b 的做法一样。
- 关系之间（`P103 vs P30`）：每条 P103 编辑和每条 P30 编辑两两比较，这里没有对角线，所以直接求平均。
- 最后打印每条曲线在第 18 个上的值，以及它的最大值出现在哪一层。

## 10. 方向由什么决定？按答案组合拆开看

在同一关系内，把两两编辑分成四类，比较各类的平均余弦相似度：
- 原答案和新答案**都相同**（例如两条都是 French → Russian）
- **只有原答案相同**（French → Russian 和 French → Dutch）
- **只有新答案相同**（French → Russian 和 English → Russian）
- **都不同**

这能告诉我们：差值方向主要由"远离原答案"决定，还是由"推向新答案"决定。

In [ ]:
def by_answer_overlap(rel, L):
    rows_r = pd.read_csv(f'{OUT}/edits_{rel}.csv')
    rows_r = rows_r[rows_r.success].set_index('case_id')
    store = torch.load(f'{OUT}/edits_{rel}.pt')
    ids = sorted(store['diff'])
    C = su.cos_matrix(torch.stack([store['diff'][i][L] for i in ids]).float())
    t = rows_r.loc[ids, 'true'].tolist()
    n = rows_r.loc[ids, 'new'].tolist()
    groups = {'same true, same new': [], 'same true only': [], 'same new only': [], 'both different': []}
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            st, sn = t[i] == t[j], n[i] == n[j]
            key = ('same true, same new' if st and sn else 'same true only' if st
                   else 'same new only' if sn else 'both different')
            groups[key].append(C[i, j].item())
    return {k: (sum(v) / len(v) if v else float('nan'), len(v)) for k, v in groups.items()}

for rel in RELATIONS:
    for L in [18, 26]:
        print(f'--- {rel}, hidden_states index {L}')
        for k, (m, cnt) in by_answer_overlap(rel, L).items():
            print(f'   {k:20} mean cos {m:6.3f}   ({cnt} pairs)')

## 11. 用自己的话写下（发给我）

1. 你的三个预测对了吗？
2. 同一关系内的相似度，比 2b 里的 0.05 高了多少？在哪一层最高？
3. 第 10 格里，四类组合的相似度怎么排序？这说明方向主要由什么决定？
4. 根据这些结果，你觉得关系内的"通用向量"有希望吗？对 P103 和 P30 分别怎么看？